# Rotary Position Embedding (RoPE)
"RoFormer: Enhanced Transformer with Rotary Position Embedding" (Su et al., 2021)

**Idea:** instead of *adding* a position vector to the token embedding, **rotate** each query and key by an angle proportional to its position. The dot product of two rotated vectors depends only on the *difference* of their angles, so attention scores see the **relative** offset $m - n$ and nothing else.

### Ways to tell a Transformer about position
| | Sinusoidal (2017) | Learned absolute (BERT, GPT-2) | Relative bias (T5, Shaw et al.) | RoPE |
|---|---|---|---|---|
| Where it enters | added to the input embedding | added to the input embedding | added to the attention **scores** | **multiplies** $q$ and $k$ in every layer |
| Encodes | absolute position | absolute position | relative offset | relative offset (through absolute rotations) |
| Parameters | 0 | $T_{max} \times D$ | buckets $\times$ heads | **0** |
| Max length | any (untrained beyond training $T$) | fixed $T_{max}$ | any (clipped / bucketed) | any |
| Changes the norm of $x$ | yes | yes | no | **no** (rotation) |
| Used by | original Transformer (`transformer.ipynb`) | `bert.ipynb`, `vit.ipynb`, GPT-2 | T5, DeBERTa | Llama, Mistral, Qwen, Gemma (`slm.ipynb`, `llama2.ipynb`), NeoBERT |

Here: the 2D case by hand, the general case in two equivalent implementations, numerical checks of its properties, and a comparison of no position / learned absolute / RoPE on the tiny Shakespeare language model.

Shape flow: `q, k (B, heads, T, hd) -> split into hd/2 pairs -> rotate pair i at position m by m * theta_i -> (B, heads, T, hd) -> q @ k^T as usual`

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

device = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(42)

## 1. The 2D case
Take a 2-dim query $q$ at position $m$ and a 2-dim key $k$ at position $n$. Rotate each by its position times a fixed angle $\theta$.

### Formula: 2D rotation
$$R(\alpha) = \begin{pmatrix} \cos\alpha & -\sin\alpha \\ \sin\alpha & \cos\alpha \end{pmatrix}, \qquad q'_m = R(m\theta)\, q, \qquad k'_n = R(n\theta)\, k$$
- $R(\alpha)$: rotation by angle $\alpha$; orthogonal, so $\lVert R(\alpha) x \rVert = \lVert x \rVert$
- $\theta$: a fixed frequency (radians per position)

### Formula: The score depends only on $m - n$
$$\langle q'_m, k'_n \rangle = q^T R(m\theta)^T R(n\theta)\, k = q^T R\big((n - m)\theta\big)\, k$$
- uses $R(\alpha)^T = R(-\alpha)$ and $R(\alpha) R(\beta) = R(\alpha + \beta)$
- shifting both tokens by the same amount $s$ leaves $n - m$, and so the score, unchanged

### Formula: Same thing with complex numbers
$$\tilde q = q_1 + i q_2, \qquad \tilde q'_m = \tilde q\, e^{i m \theta}, \qquad \langle q'_m, k'_n \rangle = \text{Re}\big[\tilde q\, \overline{\tilde k}\, e^{i (m - n) \theta}\big]$$
- a 2D rotation is multiplication by a unit complex number; this is how the Llama code implements it

In [ ]:
def rot(alpha):
    return torch.tensor([[math.cos(alpha), -math.sin(alpha)], [math.sin(alpha), math.cos(alpha)]])

theta = 0.3
q, k = torch.tensor([1.0, 0.5]), torch.tensor([0.2, 1.0])
score = lambda m, n: (rot(m * theta) @ q) @ (rot(n * theta) @ k)

print("norm before / after rotation:", q.norm().item(), (rot(7 * theta) @ q).norm().item())
print("score(m=2, n=5)    :", score(2, 5).item())
print("score(m=102, n=105):", score(102, 105).item(), " (same offset, shifted by 100)")
print("score(m=2, n=6)    :", score(2, 6).item(), " (different offset)")

fig, ax = plt.subplots(figsize=(4, 4))
for m in range(6):
    v = rot(m * theta) @ q
    ax.annotate("", xy=v.tolist(), xytext=(0, 0), arrowprops=dict(arrowstyle="->", color=plt.cm.viridis(m / 6)))
    ax.text(*(v * 1.08).tolist(), f"m={m}", fontsize=8)
ax.set_xlim(-1.4, 1.4); ax.set_ylim(-1.4, 1.4); ax.set_aspect("equal"); ax.set_title("the same q at positions 0..5")
plt.show()

## 2. General case: $d/2$ rotations at different speeds
A head's $q$ and $k$ have $d$ dims (`head_dim`). Split them into $d/2$ pairs and rotate pair $i$ with its own frequency $\theta_i$: fast for the first pairs, slow for the last.

### Formula: Rotary embedding
$$\begin{pmatrix} q'_{2i} \\ q'_{2i+1} \end{pmatrix} = \begin{pmatrix} \cos m\theta_i & -\sin m\theta_i \\ \sin m\theta_i & \cos m\theta_i \end{pmatrix} \begin{pmatrix} q_{2i} \\ q_{2i+1} \end{pmatrix}, \qquad \theta_i = 10000^{-2i/d}, \quad i = 0, \dots, d/2 - 1$$
- $q$: a query (or key) vector of one head, dim $d$
- $m$: the token's position
- $\theta_i$: the same geometric frequencies as the sinusoidal encoding; wavelengths from $2\pi$ (pair 0) to $\approx 2\pi \cdot 10000$ positions (last pair)
- as one matrix: $q'_m = R_m q$ with $R_m = \text{blockdiag}\big(R(m\theta_0), \dots, R(m\theta_{d/2-1})\big)$, a $d \times d$ orthogonal matrix
- applied to $q$ and $k$ only, **not** to $v$: position affects *where* a token looks, not *what* it reads

### Formula: Attention score with RoPE
$$\langle R_m q, R_n k \rangle = \sum_{i=0}^{d/2-1} \text{Re}\big[\tilde q_i\, \overline{\tilde k_i}\, e^{i (m - n) \theta_i}\big]$$
- $\tilde q_i = q_{2i} + i\, q_{2i+1}$: pair $i$ as a complex number
- a sum of $d/2$ "clocks" turning at different speeds; together they identify the offset $m - n$ uniquely over a very long range (like the hands of a clock, or binary digits)

Two implementations follow: the real-valued one from `slm.ipynb`, and the complex one used in Meta's Llama code.

In [ ]:
def rope_cache(T, head_dim, base=10000):
    # -> cos, sin each (T, head_dim/2)
    freqs = 1.0 / base ** (torch.arange(0, head_dim, 2).float() / head_dim)  # (head_dim/2,) theta_i
    angles = torch.arange(T).float()[:, None] * freqs[None]                  # (T, head_dim/2) m * theta_i
    return angles.cos(), angles.sin()


def apply_rope(x, cos, sin):
    # x (..., T, hd) -> (..., T, hd); rotate each pair (x_2i, x_2i+1)
    x1, x2 = x[..., 0::2], x[..., 1::2]                                      # each (..., T, hd/2)
    return torch.stack([x1 * cos - x2 * sin, x1 * sin + x2 * cos], dim=-1).flatten(-2)


def apply_rope_complex(x, T, base=10000):
    # same rotation as complex multiplication: (x_2i + i x_2i+1) * exp(i m theta_i)
    hd = x.shape[-1]
    freqs = 1.0 / base ** (torch.arange(0, hd, 2).float() / hd)              # (hd/2,)
    angles = torch.outer(torch.arange(T).float(), freqs)                     # (T, hd/2)
    rot = torch.polar(torch.ones_like(angles), angles)                       # (T, hd/2) complex, e^{i m theta_i}
    xc = torch.view_as_complex(x.reshape(*x.shape[:-1], -1, 2))              # (..., T, hd/2) complex
    return torch.view_as_real(xc * rot).flatten(-2)                          # (..., T, hd)


def rope_matrix(m, head_dim, base=10000):
    # explicit R_m (hd, hd): block-diagonal 2x2 rotations, for checking only
    freqs = 1.0 / base ** (torch.arange(0, head_dim, 2).float() / head_dim)
    return torch.block_diag(*[rot(m * f.item()) for f in freqs])


T, hd = 64, 16
cos, sin = rope_cache(T, hd)
x = torch.randn(2, 4, T, hd)                                                 # (B, heads, T, hd)
a, b = apply_rope(x, cos, sin), apply_rope_complex(x, T)
print("real vs complex implementation, max diff:", (a - b).abs().max().item())
print("vs explicit R_m at m = 9, max diff      :", (a[0, 0, 9] - rope_matrix(9, hd) @ x[0, 0, 9]).abs().max().item())

## 3. Properties, checked numerically
1. **Norm preserving**: rotations do not change vector length (additive encodings do).
2. **Relative**: $\langle R_m q, R_n k \rangle$ is unchanged if both positions shift by $s$.
3. **Position 0 is the identity**: $R_0 = I$.
4. For contrast, an **additive** sinusoidal encoding $\langle q + p_m, k + p_n \rangle$ is *not* shift-invariant: it contains cross terms $\langle q, p_n \rangle$ and $\langle p_m, k \rangle$ that depend on absolute position.

In [ ]:
T, hd = 256, 64
cos, sin = rope_cache(T, hd)
q, k = torch.randn(hd), torch.randn(hd)
Q = apply_rope(q.expand(T, hd), cos, sin)        # (T, hd): the same q placed at every position
K = apply_rope(k.expand(T, hd), cos, sin)        # (T, hd)
S = Q @ K.T                                      # (T, T): S[m, n] = <R_m q, R_n k>

print("1. norms:", q.norm().item(), "->", Q.norm(dim=-1).min().item(), "..", Q.norm(dim=-1).max().item())
print("2. S[3, 10], S[53, 60], S[203, 210]:", S[3, 10].item(), S[53, 60].item(), S[203, 210].item())
print("   max variation along each diagonal (fixed m - n):", max((S.diagonal(o).max() - S.diagonal(o).min()).item() for o in range(-50, 51)))
print("3. R_0 q == q:", torch.allclose(Q[0], q))

def sinusoidal(T, dim, base=10000):
    # (T, dim) additive encoding of the original Transformer
    freqs = 1.0 / base ** (torch.arange(0, dim, 2).float() / dim)
    angles = torch.arange(T).float()[:, None] * freqs[None]
    return torch.stack([angles.sin(), angles.cos()], dim=-1).flatten(-2)

P = sinusoidal(T, hd)
S_add = (q + P) @ (k + P).T                      # (T, T)
print("4. additive: S[3, 10], S[53, 60], S[203, 210]:", S_add[3, 10].item(), S_add[53, 60].item(), S_add[203, 210].item())

fig, ax = plt.subplots(1, 2, figsize=(10, 4))
ax[0].imshow(S[:64, :64], cmap="RdBu"); ax[0].set_title("RoPE: constant along diagonals")
ax[1].imshow(S_add[:64, :64], cmap="RdBu"); ax[1].set_title("additive sinusoidal: not")
for a in ax: a.set_xlabel("key position n"); a.set_ylabel("query position m")
plt.show()

## 4. What the frequencies look like
Left: the rotation angle $m\theta_i$ (mod $2\pi$) for every position and pair. The first pairs spin fast (fine position), the last barely move over 256 positions (coarse position).

Right: the score between two **identical** vectors ($q = k = \mathbf{1}$) as a function of their distance. It is largest at distance 0 and decays, with oscillations: content being equal, RoPE biases attention toward nearby tokens (the paper's "long-term decay").

### Formula: Score of identical all-ones vectors
$$\langle R_m \mathbf{1}, R_n \mathbf{1} \rangle = 2 \sum_{i=0}^{d/2-1} \cos\big((m - n)\,\theta_i\big)$$
- equals $d$ at $m = n$; for large $|m - n|$ the cosines fall out of phase and partly cancel

In [ ]:
T, hd = 256, 64
freqs = 1.0 / 10000 ** (torch.arange(0, hd, 2).float() / hd)
angles = torch.arange(T).float()[:, None] * freqs[None]                 # (T, hd/2)
dist = torch.arange(T).float()
decay = 2 * torch.cos(dist[:, None] * freqs[None]).sum(-1)              # (T,)

fig, ax = plt.subplots(1, 2, figsize=(12, 3.5))
im = ax[0].imshow((angles % (2 * math.pi)).T, aspect="auto", cmap="twilight")
ax[0].set_xlabel("position m"); ax[0].set_ylabel("pair i"); ax[0].set_title(r"$m\theta_i$ mod $2\pi$"); plt.colorbar(im, ax=ax[0])
ax[1].plot(dist, decay); ax[1].set_xlabel("distance |m - n|"); ax[1].set_ylabel("score"); ax[1].set_title("q = k = 1")
plt.show()
print("wavelength of pair 0:", (2 * math.pi / freqs[0]).item(), "positions | last pair:", (2 * math.pi / freqs[-1]).item())

## 5. Does it help? Three language models
The decoder from `slm.ipynb` (RMSNorm, SwiGLU, causal attention) on character-level tiny Shakespeare, with only the position mechanism changed:
- **none**: no position information at all. A causal decoder is not fully order-blind (token $t$ sees exactly $t$ predecessors), but it cannot tell *which* of them was the previous character.
- **learned absolute**: `nn.Embedding(block_size, D)` added to the token embedding (GPT-2 / BERT style).
- **RoPE**: rotate $q, k$ in every layer.

Smaller than `slm.ipynb` ($D = 128$, 4 layers, 2000 steps) so three runs stay quick.

In [ ]:
text = open("../datasets/tiny_shakespeare.txt").read()
chars = sorted(set(text))
vocab_size = len(chars)
stoi = {c: i for i, c in enumerate(chars)}
encode = lambda s: [stoi[c] for c in s]

data = torch.tensor(encode(text))          # (1115394,)
n = int(0.9 * len(data))
train_data, val_data = data[:n], data[n:]

block_size = 128   # T
batch_size = 32    # B

def get_batch(split, T=block_size):
    # -> x (B, T), y (B, T); y is x shifted left by one
    d = train_data if split == "train" else val_data
    ix = torch.randint(len(d) - T - 1, (batch_size,))
    x = torch.stack([d[i:i + T] for i in ix])
    y = torch.stack([d[i + 1:i + T + 1] for i in ix])
    return x.to(device), y.to(device)


class RMSNorm(nn.Module):
    # (..., D) -> (..., D)
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        return x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.eps) * self.weight


class CausalSelfAttention(nn.Module):
    # (B, T, D) -> (B, T, D); RoPE applied only if cos/sin are given
    def __init__(self, dim, num_heads):
        super().__init__()
        self.num_heads = num_heads
        self.head_dim = dim // num_heads
        self.qkv = nn.Linear(dim, 3 * dim, bias=False)
        self.proj = nn.Linear(dim, dim, bias=False)

    def forward(self, x, cos=None, sin=None):
        B, T, D = x.shape
        qkv = self.qkv(x).reshape(B, T, 3, self.num_heads, self.head_dim)  # (B, T, 3, heads, hd)
        q, k, v = qkv.permute(2, 0, 3, 1, 4)                                # each (B, heads, T, hd)
        if cos is not None:
            q, k = apply_rope(q, cos, sin), apply_rope(k, cos, sin)         # the only line RoPE adds

        att = (q @ k.transpose(-2, -1)) / self.head_dim ** 0.5              # (B, heads, T, T)
        mask = torch.ones(T, T, dtype=torch.bool, device=x.device).tril()
        att = att.masked_fill(~mask, float("-inf")).softmax(dim=-1)
        return self.proj((att @ v).transpose(1, 2).reshape(B, T, D))


class SwiGLU(nn.Module):
    # (..., D) -> (..., D)
    def __init__(self, dim, hidden):
        super().__init__()
        self.w1 = nn.Linear(dim, hidden, bias=False)
        self.w3 = nn.Linear(dim, hidden, bias=False)
        self.w2 = nn.Linear(hidden, dim, bias=False)

    def forward(self, x):
        return self.w2(F.silu(self.w1(x)) * self.w3(x))


class Block(nn.Module):
    # (B, T, D) -> (B, T, D)
    def __init__(self, dim, num_heads):
        super().__init__()
        self.norm1, self.attn = RMSNorm(dim), CausalSelfAttention(dim, num_heads)
        self.norm2, self.ffn = RMSNorm(dim), SwiGLU(dim, 64 * ((8 * dim // 3 + 63) // 64))

    def forward(self, x, cos, sin):
        x = x + self.attn(self.norm1(x), cos, sin)
        return x + self.ffn(self.norm2(x))


class LM(nn.Module):
    # idx (B, T) -> logits (B, T, V); pos in {"none", "learned", "rope"}
    def __init__(self, pos, max_len=1024, dim=128, depth=4, num_heads=4):
        super().__init__()
        self.pos = pos
        self.embed = nn.Embedding(vocab_size, dim)
        if pos == "learned":
            self.pos_embed = nn.Embedding(block_size, dim)     # exists only for positions seen in training
        self.blocks = nn.ModuleList([Block(dim, num_heads) for _ in range(depth)])
        self.norm = RMSNorm(dim)
        self.head = nn.Linear(dim, vocab_size, bias=False)
        self.head.weight = self.embed.weight
        self.apply(lambda m: nn.init.normal_(m.weight, std=0.02) if isinstance(m, (nn.Linear, nn.Embedding)) else None)
        cos, sin = rope_cache(max_len, dim // num_heads)       # no parameters, any length
        self.register_buffer("cos", cos)
        self.register_buffer("sin", sin)

    def forward(self, idx):
        T = idx.shape[1]
        x = self.embed(idx)                                                 # (B, T, D)
        if self.pos == "learned":
            x = x + self.pos_embed(torch.arange(T, device=idx.device))      # (B, T, D) + (T, D)
        cos, sin = (self.cos[:T], self.sin[:T]) if self.pos == "rope" else (None, None)
        for block in self.blocks:
            x = block(x, cos, sin)
        return self.head(self.norm(x))                                      # (B, T, V)


@torch.no_grad()
def estimate_loss(model, split="val", T=block_size, iters=50):
    model.eval()
    out = sum(F.cross_entropy(model(x).reshape(-1, vocab_size), y.reshape(-1)).item() for x, y in (get_batch(split, T) for _ in range(iters))) / iters
    model.train()
    return out


max_iters = 2000
eval_every = 250
models, curves = {}, {}
for pos in ["none", "learned", "rope"]:
    torch.manual_seed(42)
    model = LM(pos).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)
    curves[pos] = []
    for it in range(max_iters + 1):
        if it % eval_every == 0:
            curves[pos].append((it, estimate_loss(model)))
        if it == max_iters:
            break
        x, y = get_batch("train")
        loss = F.cross_entropy(model(x).reshape(-1, vocab_size), y.reshape(-1))   # (B*T, V) vs (B*T,)
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
    models[pos] = model
    print(f"{pos:8s} | params {sum(p.numel() for p in model.parameters()) / 1e6:.3f}M | val loss {curves[pos][-1][1]:.4f}")

for pos, c in curves.items():
    plt.plot(*zip(*c), marker="o", ms=3, label=pos)
plt.xlabel("iter"); plt.ylabel("val loss"); plt.legend(); plt.show()

## 6. Longer sequences than seen in training
All models were trained with $T = 128$. Evaluate the per-character loss with longer contexts.
- **learned absolute** has no embedding for position 128 and beyond: it cannot run at all.
- **RoPE** has no table, so it runs at any length. Whether the loss stays good is a separate question: offsets larger than 127 were never seen in training.

In [ ]:
for T in [64, 128, 256, 512]:
    row = [f"T = {T:4d}"]
    for pos in ["none", "learned", "rope"]:
        if pos == "learned" and T > block_size:
            row.append(f"{pos}: no embedding for pos >= {block_size}")
        else:
            row.append(f"{pos}: {estimate_loss(models[pos], T=T, iters=20):.4f}")
    print(" | ".join(row))

## Notes
- RoPE is the default in current LLMs (Llama, Mistral, Qwen, Gemma, DeepSeek) and in modern encoders (ModernBERT, NeoBERT). It adds no parameters, needs no change to the input, and is one extra line in attention.
- "Relative" refers to the attention **score**. The rotation itself uses the absolute position $m$; the absolute part cancels in the dot product.
- Length extrapolation is better than with absolute embeddings but not free (in §6 the RoPE loss rises once $T$ exceeds 128, it just rises from a model that still runs): beyond the training length the model meets rotation angles it has never seen. Long-context models are trained (or fine-tuned) at long lengths, or rescale the frequencies (position interpolation, NTK-aware scaling, **YaRN**). Llama 3 raised the base from 10,000 to 500,000 so the slow pairs cover longer contexts.
- With a KV cache (`llama2.ipynb`), cached keys are already rotated by their own position and never need updating: only the new token's $q, k$ are rotated.
- For images, RoPE is extended to 2D by rotating half the pairs by the patch's $x$ coordinate and half by its $y$ (used in recent ViTs and in Llama 4's vision encoder).
- Alternatives in the same family: **ALiBi** (subtract a linear penalty $\propto |m - n|$ from the scores, no rotation) and T5's learned relative bias buckets.